# Phase 2: LLM Benchmarking for Rare Disease Diagnosis

**Portfolio Project** | AI × Healthcare Research  
**Goal**: Systematically evaluate Claude's ability to diagnose rare diseases from clinical case summaries  
**Key question**: Does LLM accuracy vary by disease rarity tier?

---

## 1. Setup

In [1]:
# Install dependencies (run once)
# !pip install anthropic tqdm

In [2]:
import os
import re
import json
import time
import random
import pandas as pd
import numpy as np
import anthropic
from tqdm import tqdm
from pathlib import Path

# ── API key ────────────────────────────────────────────────────────────────
# Set your key in the environment before running:
#   export ANTHROPIC_API_KEY="sk-ant-..."
# Or uncomment and paste directly (never commit to git!):
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from env
MODEL  = "claude-sonnet-4-20250514"

print(f"Model  : {MODEL}")
print(f"API key: {'✓ set' if os.environ.get('ANTHROPIC_API_KEY') else '✗ NOT SET'}")

Model  : claude-sonnet-4-20250514
API key: ✗ NOT SET


In [ ]:
# Load datasets
df_train = pd.read_csv("train_split_50k_cases.csv")
df_eval  = pd.read_csv("Evaluation.csv")

# Build rarity tiers from training set frequency
disease_freq = df_train["Disease"].value_counts()

def rarity_tier(disease_name):
    """Map a disease name to a rarity tier based on training frequency."""
    freq = disease_freq.get(disease_name, 0)
    if freq == 0:  return "Zero-shot (novel)"
    if freq == 1:  return "Ultra-rare (1)"
    if freq <= 5:  return "Very rare (2–5)"
    if freq <= 20: return "Rare (6–20)"
    if freq <= 100:return "Moderate (21–100)"
    return "Common (100+)"

df_eval["rarity_tier"] = df_eval["Disease"].apply(rarity_tier)
print("Eval set rarity tier distribution:")
print(df_eval["rarity_tier"].value_counts())

## 2. Prompt Design

Good prompt design is itself a research decision. We test two strategies:
- **Direct**: Ask for a single best answer
- **Differential**: Ask for top-5 candidates (mirrors real clinical reasoning)

In [ ]:
SYSTEM_PROMPT = """You are an expert clinical diagnostician with deep knowledge of rare diseases.
Your task is to identify the most likely rare disease diagnosis from a clinical case summary.
Be precise — use the standard disease name as it would appear in a medical database.
Do not add explanations unless asked. Respond only in the JSON format requested."""

def build_top1_prompt(case_summary: str) -> str:
    """Prompt for single best diagnosis."""
    # Truncate very long cases to ~3000 chars to stay within context efficiently
    summary = case_summary[:3000] if len(case_summary) > 3000 else case_summary
    return f"""Based on the following clinical case summary, identify the single most likely rare disease diagnosis.

Clinical case:
{summary}

Respond with ONLY valid JSON in this exact format:
{{"diagnosis": "<disease name>"}}"""

def build_top5_prompt(case_summary: str) -> str:
    """Prompt for differential diagnosis (top 5)."""
    summary = case_summary[:3000] if len(case_summary) > 3000 else case_summary
    return f"""Based on the following clinical case summary, provide a differential diagnosis with the 5 most likely rare diseases, ranked from most to least likely.

Clinical case:
{summary}

Respond with ONLY valid JSON in this exact format:
{{"diagnoses": ["<1st>", "<2nd>", "<3rd>", "<4th>", "<5th>"]}}"""

print("Prompts defined.")

## 3. Matching Strategy

LLM outputs won't always exactly match the ground truth disease name. We use three levels:
- **Exact match**: string identical (after lowercasing/stripping)
- **Synonym match**: LLM output appears in the disease's `Synonyms` field
- **Fuzzy match**: Jaccard similarity on word tokens ≥ 0.6 (catches word-order variants)

In [ ]:
def normalize(text: str) -> str:
    """Lowercase, strip punctuation, trim whitespace."""
    return re.sub(r"[^a-z0-9 ]", "", text.lower()).strip()

def jaccard_similarity(a: str, b: str) -> float:
    """Word-level Jaccard similarity between two strings."""
    sa, sb = set(normalize(a).split()), set(normalize(b).split())
    if not sa or not sb: return 0.0
    return len(sa & sb) / len(sa | sb)

def is_match(prediction: str, ground_truth: str, synonyms: str = "") -> dict:
    """
    Returns a dict with match type and boolean hit.
    match_type: 'exact' | 'synonym' | 'fuzzy' | 'none'
    """
    pred_norm = normalize(prediction)
    gt_norm   = normalize(ground_truth)

    # Exact
    if pred_norm == gt_norm:
        return {"hit": True, "match_type": "exact"}

    # Synonym
    if pd.notna(synonyms) and synonyms:
        for syn in str(synonyms).split(";"):
            if pred_norm == normalize(syn.strip()):
                return {"hit": True, "match_type": "synonym"}

    # Fuzzy
    if jaccard_similarity(prediction, ground_truth) >= 0.6:
        return {"hit": True, "match_type": "fuzzy"}

    return {"hit": False, "match_type": "none"}

# Quick sanity checks
assert is_match("Cystic Fibrosis", "Cystic fibrosis")["hit"] == True
assert is_match("CF", "Cystic fibrosis", synonyms="CF; Mucoviscidosis")["hit"] == True
assert is_match("Unrelated disease", "Cystic fibrosis")["hit"] == False
print("Matching functions validated.")

## 4. API Call Wrapper with Rate-Limit Handling

In [ ]:
def call_claude(prompt: str, max_retries: int = 3, pause: float = 2.0) -> str | None:
    """
    Call the Claude API and return the raw text response.
    Retries on rate-limit errors with exponential backoff.
    Returns None if all retries fail.
    """
    for attempt in range(max_retries):
        try:
            message = client.messages.create(
                model=MODEL,
                max_tokens=256,          # Diagnoses are short
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": prompt}],
            )
            return message.content[0].text.strip()

        except anthropic.RateLimitError:
            wait = pause * (2 ** attempt)
            print(f"  Rate limit hit — waiting {wait:.0f}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)

        except anthropic.APIError as e:
            print(f"  API error: {e}")
            time.sleep(pause)

    return None


def parse_json_response(raw: str | None, key: str) -> list[str]:
    """
    Parse JSON from model output. Returns a list (always) for uniform handling.
    key: 'diagnosis' (top-1) or 'diagnoses' (top-5)
    """
    if raw is None:
        return []
    try:
        # Strip markdown code fences if model added them
        cleaned = re.sub(r"```(?:json)?\s*", "", raw).strip().rstrip("`")
        data = json.loads(cleaned)
        val  = data.get(key, [])
        return [val] if isinstance(val, str) else list(val)
    except (json.JSONDecodeError, AttributeError):
        # Fallback: grab first quoted string
        matches = re.findall(r'"([^"]{4,}?)"', raw)
        return matches[:1] if matches else []

print("API wrapper defined.")

## 5. Sampling Strategy

Running all 6,915 eval cases is costly (~$5–15 depending on case length). For a portfolio, a **stratified sample** is better: it ensures every rarity tier is fairly represented and reduces cost by 10–20x.

We draw **25 cases per tier** (≤ 150 total). This is enough for statistically meaningful comparisons.

In [ ]:
CASES_PER_TIER = 25
RANDOM_SEED    = 42

sampled = (
    df_eval
    .groupby("rarity_tier", group_keys=False)
    .apply(lambda g: g.sample(min(CASES_PER_TIER, len(g)), random_state=RANDOM_SEED))
    .reset_index(drop=True)
)

print(f"Total cases to evaluate: {len(sampled)}")
print(sampled["rarity_tier"].value_counts())

# Rough cost estimate (claude-sonnet: ~$3 per MTok input, ~$15 per MTok output)
avg_chars   = sampled["CaseSummary"].str[:3000].str.len().mean()
est_tokens  = avg_chars / 4  # ~4 chars per token
est_cost    = len(sampled) * (est_tokens * 3e-6 + 64 * 15e-6)
print(f"\nEstimated cost: ~${est_cost:.2f} USD")

## 6. Run the Benchmark

⚠️ **This cell makes real API calls and costs money.** The checkpoint system saves results after every case so you can resume if interrupted.

In [ ]:
CHECKPOINT_FILE = "benchmark_results.jsonl"
INTER_CALL_DELAY = 0.5  # seconds between calls (be a good API citizen)

# Load existing checkpoint if present
completed_ids = set()
if Path(CHECKPOINT_FILE).exists():
    with open(CHECKPOINT_FILE) as f:
        for line in f:
            try:
                completed_ids.add(json.loads(line)["case_id"])
            except Exception:
                pass
    print(f"Resuming from checkpoint: {len(completed_ids)} cases already done")

todo = sampled[~sampled["Unnamed: 0"].isin(completed_ids)]
print(f"Cases remaining: {len(todo)}")

In [ ]:
with open(CHECKPOINT_FILE, "a") as f:
    for _, row in tqdm(todo.iterrows(), total=len(todo), desc="Benchmarking"):
        case_id    = row["Unnamed: 0"]
        summary    = row["CaseSummary"]
        ground_truth = row["Disease"]
        synonyms   = row.get("Synonyms", "")
        tier       = row["rarity_tier"]

        # ── Top-1 prediction ──────────────────────────────────────────────
        raw_top1   = call_claude(build_top1_prompt(summary))
        preds_top1 = parse_json_response(raw_top1, "diagnosis")
        top1_pred  = preds_top1[0] if preds_top1 else ""
        top1_result = is_match(top1_pred, ground_truth, synonyms)

        time.sleep(INTER_CALL_DELAY)

        # ── Top-5 prediction ──────────────────────────────────────────────
        raw_top5   = call_claude(build_top5_prompt(summary))
        preds_top5 = parse_json_response(raw_top5, "diagnoses")
        top5_hit   = any(is_match(p, ground_truth, synonyms)["hit"] for p in preds_top5)

        time.sleep(INTER_CALL_DELAY)

        record = {
            "case_id":       case_id,
            "rarity_tier":   tier,
            "ground_truth":  ground_truth,
            "top1_pred":     top1_pred,
            "top1_hit":      top1_result["hit"],
            "top1_match_type": top1_result["match_type"],
            "top5_preds":    preds_top5,
            "top5_hit":      top5_hit,
            "text_length":   len(summary),
        }
        f.write(json.dumps(record) + "\n")
        f.flush()

print("\n✓ Benchmarking complete. Results saved to:", CHECKPOINT_FILE)

## 7. Load & Analyse Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 130, "font.size": 11})

# Load results
records = []
with open(CHECKPOINT_FILE) as f:
    for line in f:
        try: records.append(json.loads(line))
        except Exception: pass

df_results = pd.DataFrame(records)
print(f"Loaded {len(df_results)} results")
df_results.head(3)

In [ ]:
# ── Accuracy by rarity tier ────────────────────────────────────────────────
TIER_ORDER = [
    "Zero-shot (novel)",
    "Ultra-rare (1)",
    "Very rare (2–5)",
    "Rare (6–20)",
    "Moderate (21–100)",
    "Common (100+)",
]

tier_stats = (
    df_results
    .groupby("rarity_tier")
    .agg(
        n       =("top1_hit", "count"),
        top1_acc=("top1_hit", "mean"),
        top5_acc=("top5_hit", "mean"),
    )
    .reindex([t for t in TIER_ORDER if t in df_results["rarity_tier"].unique()])
    .reset_index()
)

print(tier_stats.to_string(index=False))

In [ ]:
# ── Figure: Top-1 vs Top-5 accuracy by rarity tier ───────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

x     = np.arange(len(tier_stats))
width = 0.35

bars1 = ax.bar(x - width/2, tier_stats["top1_acc"] * 100, width,
               label="Top-1 accuracy", color="#4c72b0", alpha=0.9)
bars2 = ax.bar(x + width/2, tier_stats["top5_acc"] * 100, width,
               label="Top-5 accuracy", color="#dd8452", alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(tier_stats["rarity_tier"], rotation=20, ha="right")
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(0, 100)
ax.set_title("Claude Diagnostic Accuracy by Disease Rarity Tier", fontsize=13)
ax.legend()

# Annotate n per tier
for i, row in tier_stats.iterrows():
    ax.text(i, -8, f"n={row['n']}", ha="center", fontsize=9,
            color="gray", transform=ax.get_xaxis_transform())

plt.tight_layout()
plt.savefig("fig_accuracy_by_tier.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Figure: Match type breakdown (exact / synonym / fuzzy / none) ─────────
match_type_counts = (
    df_results
    .groupby(["rarity_tier", "top1_match_type"])
    .size()
    .unstack(fill_value=0)
    .reindex([t for t in TIER_ORDER if t in df_results["rarity_tier"].unique()])
)

match_type_counts_pct = match_type_counts.div(match_type_counts.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
colors = {"exact": "#2ca02c", "synonym": "#98df8a", "fuzzy": "#aec7e8", "none": "#d62728"}

bottom = np.zeros(len(match_type_counts_pct))
for mtype in ["exact", "synonym", "fuzzy", "none"]:
    if mtype in match_type_counts_pct.columns:
        vals = match_type_counts_pct[mtype].values
        ax.bar(match_type_counts_pct.index, vals, bottom=bottom,
               label=mtype.capitalize(), color=colors[mtype], alpha=0.9)
        bottom += vals

ax.set_xlabel("Rarity tier")
ax.set_ylabel("% of cases")
ax.set_title("Match Type Breakdown by Rarity Tier (Top-1 Prediction)")
ax.legend(loc="upper right")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("fig_match_type_breakdown.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Text length effect ────────────────────────────────────────────────────
df_results["length_bucket"] = pd.cut(
    df_results["text_length"],
    bins=[0, 500, 1500, 3000, 100000],
    labels=["<500 chars", "500–1.5k", "1.5k–3k", "3k+ (truncated)"]
)

len_stats = (
    df_results
    .groupby("length_bucket", observed=True)
    .agg(n=("top1_hit", "count"), top1_acc=("top1_hit", "mean"))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(len_stats["length_bucket"], len_stats["top1_acc"] * 100,
       color=sns.color_palette("Blues_d", len(len_stats)))
ax.set_ylabel("Top-1 Accuracy (%)")
ax.set_xlabel("Case summary length")
ax.set_title("Accuracy vs. Case Summary Length")
for i, row in len_stats.iterrows():
    ax.text(i, row["top1_acc"] * 100 + 1, f"n={row['n']}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("fig_accuracy_by_length.png", bbox_inches="tight")
plt.show()

## 8. Summary Table — Paper-Ready Results

In [ ]:
# 95% confidence interval for proportions (Wilson interval)
from scipy.stats import norm

def wilson_ci(hits, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p = hits / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    margin = (z * np.sqrt(p*(1-p)/n + z**2/(4*n**2))) / denom
    return (max(0, center - margin), min(1, center + margin))

rows = []
for _, row in tier_stats.iterrows():
    hits = int(row["top1_acc"] * row["n"])
    lo, hi = wilson_ci(hits, row["n"])
    rows.append({
        "Rarity tier":    row["rarity_tier"],
        "N cases":        row["n"],
        "Top-1 acc (%)": f"{row['top1_acc']*100:.1f}",
        "95% CI":        f"[{lo*100:.1f}, {hi*100:.1f}]",
        "Top-5 acc (%)": f"{row['top5_acc']*100:.1f}",
    })

df_table = pd.DataFrame(rows)
print(df_table.to_string(index=False))
df_table.to_csv("results_summary_table.csv", index=False)
print("\nSaved to results_summary_table.csv")

## 9. Key Findings & Interpretation

Fill this in after your run. Template:

---

### Preliminary findings

- **Top-1 accuracy** ranged from __%  (ultra-rare tier) to __% (common tier), confirming the hypothesis that LLMs perform significantly worse on ultra-rare diseases.
- **Top-5 accuracy** showed a similar gradient, suggesting the correct diagnosis is present in the model's "uncertainty cloud" but not ranked first.
- **Match type analysis**: For correct predictions, __% were exact matches, __% synonym matches, and __% fuzzy — indicating the model uses alternative disease names frequently.
- **Text length effect**: Cases truncated to 3k characters showed [better/worse/similar] accuracy to short cases, suggesting [conclusion].

### Clinical implication
The performance gap between common and ultra-rare diseases is clinically significant. Diseases with the fewest training examples are also the diseases where diagnostic error causes the most harm — patients with ultra-rare diseases typically experience years of diagnostic delay before receiving a correct diagnosis.

### Next steps → Phase 3
- [ ] Prompt sensitivity: does rephrasing improve ultra-rare accuracy?
- [ ] Confidence calibration: is the model overconfident on wrong ultra-rare predictions?
- [ ] ICD chapter breakdown: which disease *domains* are hardest?
- [ ] Knowledge-grounded prompting: add OMIM/GARD context and retest